# Phase 4: Perceptual Cloud Visualization

This notebook visualizes the **Perceptual Cloud** (The Batter's Eye).

*   **Cyan Cloud (Physical):** Where the ball effectively *is* (Physics + Execution error).
*   **Green Cloud (Perceptual):** Where the batter *thinks* it is (Bayesian Update).

The Green cloud starts large (high visual uncertainty) and shrinks rapidly as the ball approaches the plate.

In [1]:
%load_ext autoreload
%autoreload 2

import sys
import os
sys.path.append(os.path.abspath('../'))

import pandas as pd
import numpy as np
from src.data_handler import StatcastDataHandler
from src.physics import calculate_trajectory
from src.uncertainty import generate_pitch_cloud
from src.perception import PerceptualCloud
from src.visualization import visualize_pitch_cloud

In [2]:
# 1. Load Data
handler = StatcastDataHandler()
df = handler.fetch_statcast(start_year=2024, end_year=2024)
print(f"Loaded {len(df)} rows")

Loading data from 2024 to 2024...
Loaded 722296 rows


In [3]:
# 2. Select Pitcher (Corbin Burnes)
target_name = 'Snell, Blake'

try:
    top_pitcher = df[df['player_name'] == target_name]['pitcher'].iloc[0]
except IndexError:
    print(f"{target_name} not found, falling back to mode.")
    top_pitcher = df['pitcher'].mode()[0]

pitcher_df = df[df['pitcher'] == top_pitcher]
pitcher_name = pitcher_df['player_name'].iloc[0]
print(f"Selected Pitcher: {pitcher_name}")

# Select Top Pitch Type
pitch_type = pitcher_df['pitch_type'].mode()[0]
row = pitcher_df[pitcher_df['pitch_type'] == pitch_type].iloc[0]
print(f"Selected Pitch: {pitch_type}")

Selected Pitcher: Snell, Blake
Selected Pitch: FF


In [4]:
# 3. Run Simulation (Physical + Perceptual)
print("Generating Physical Cloud (Cyan)...")
cloud_trajs = generate_pitch_cloud(row, n_samples=200)
actual_traj = calculate_trajectory(row)

print("Generating Perceptual Cloud (Green) - Simulating Eye...")
eye_model = PerceptualCloud()
perceptual_trajs = eye_model.calculate_posterior_cloud(cloud_trajs, actual_traj)

print("Done.")

Generating Physical Cloud (Cyan)...
Generating Perceptual Cloud (Green) - Simulating Eye...
Scoring: Computing Bayesian Perception...
Done.


In [5]:
# 4. Visualize
fig = visualize_pitch_cloud(actual_traj, cloud_trajs, 
                            pitch_type=f"{pitcher_name} {pitch_type} Analysis", 
                            perceptual_trajs=perceptual_trajs)
fig.show()

In [6]:
# --- 4.1 ULTRA CLEAN VISUALIZATION ---
import importlib
import src.visualization
importlib.reload(src.visualization)
from src.visualization import visualize_pitch_cloud

print("Rendering Ultra Clean Cloud (Perceptual Only)...")

fig_clean = visualize_pitch_cloud(
    actual_traj, 
    cloud_trajs, 
    pitch_type=f"{pitcher_name} {pitch_type} (Perceptual Only)", 
    perceptual_trajs=perceptual_trajs,
    show_paths=False,      # Hides Lines & Dots
    show_physical=False    # Hides Cyan Mesh
)

fig_clean.show()

Rendering Ultra Clean Cloud (Perceptual Only)...
